# Point mutations using mBuild: swap side chain chemistry, keeping the backbone coordinates
### Joseph R. Laforet Jr.

`Protein.mutate` replaces the side chain of one residue and leaves the
backbone coordinates untouched. Name the residue and the
new side chain, and mBuild removes everything beyond the CA–CB bond, builds the
new side chain from a Chemical Component Dictionary (CCD) component or from a
fragment you wrote, bonds it to CA, renames the residue, re-matches its
definition, and relaxes only the new atoms.

The example is a real-world modified protein. Fluorescence studies of human
fibronectin introduce **p-azido-L-phenylalanine (AzF) at position 1381** and a
**cysteine at position 1500**, numbered as in full-length fibronectin, so that
a dye can be clicked onto the azide and a second dye attached to the thiol.
Both positions are serines in the wild type, and both lie in the FN7–10
fragment of PDB entry **1FNF**, which is what this notebook loads.

The file `1fnf_clean.pdb` was prepared by: downloading from the PDB, removing crystallographic waters, and finally adding in hydrogens via PyMol. The code was written before this example, so the file also serves as a test of the loader on a structure it had never seen.

This notebook teaches three things about the module:

- The `CCDLibrary` is the set of side chains `mutate` knows. The loader,
  `fragment_from_ccd` and `mutate` all read the same library.
- Chemistry outside the library is written as a fragment. The call is the
  same; what differs is that the residue ends with `template = None`.
- A residue without a template is written fine, but no template reader can
  read it back until it is given a definition. Notebook 4 shows how.

In the review stack this is PR 7, `mutate`.

In [ ]:
import logging
import time
import warnings

warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

import ipywidgets
import mbuild as mb
import pandas as pd
from rdkit.Chem import Draw

from mbuild.biopolymers import CCDLibrary, Protein, fragment_from_ccd, prepare_fragment
from demo_utils import alpha_carbon_cip, residue_to_rdkit, show_protein

logging.getLogger("mbuild").setLevel(logging.WARNING)

wild_type = Protein("1fnf_clean.pdb", download=True)
print(
    f"{wild_type.n_particles} atoms, {len(list(wild_type.residues()))} residues, "
    f"chains {[c.chain_id for c in wild_type.chains]}, net formal charge {wild_type.net_formal_charge}"
)
for resnum in (1381, 1500):
    residue = wild_type.get_residue(resnum, chain_id="A")
    print(resnum, residue.name, residue.template.description, sorted(p.name for p in residue.particles()))

**About the warning.** PyMol added the hydrogens and named them `H01`,
`H02`, … in file order, and no residue definition carries those names. mBuild
places each such hydrogen on the heavy atom nearest to it and builds it under
the CCD name, so `H01` of the first proline became `HA`. The rule is for
hydrogens only: a heavy atom with an unknown name still refuses, because its
name states a chemistry that a distance cannot confirm. A strict
template-matching reader such as OpenFF Pablo refuses the raw file outright,
and it reads the file mBuild writes back out, because that file carries the
CCD names. 

`download=True` lets the library fetch a CCD component that is not bundled.
The 20 canonical amino acids ship with mBuild; the AzF component below is
fetched once and cached.

## Which side chains can you mutate to?

`mutate` takes its side chains from a `CCDLibrary`, the same object the
loader matched the residues against. `library[code]` is the list of
`ResidueTemplate` variants for one component, and `fragment_from_ccd(code,
"CA")` cuts a side chain out of one. So the set of side chains you can
mutate to is the set of components the library can supply.

If you want to stay in predefined CCD templates, you can mutate to any CCD component of type *L-peptide linking* or *D-peptide linking* with one side chain on its alpha carbon. The side-chain atom is found by its
bond to `CA`, not by its name, because CCD names outside the canonical residues
follow no convention (`4AF` below calls its beta carbon `C3`). Atom names,
elements, bonds, bond orders, formal charges and ideal coordinates come from the
component, so the mutant carries the same names a residue library expects.

If you are interested in modelling a chemistry that is not supported by this workflow, first try the custom fragment approach later in the notebook, then please raise an issue if that still doesn't work.

Two rules:

- **Glycine** has no `CB`. Mutating *to* glycine adds an alpha hydrogen in
  place of the side chain; mutating *from* glycine picks the alpha hydrogen
  that gives the handedness you ask for (L by default).
- **Proline** is refused in both directions. Its side chain bonds the backbone
  nitrogen, so removing or adding it would change the backbone that `mutate`
  promises to keep. NOTE: we can support this if needed

Here are the 20 bundled canonical amino acids. The table is built from the same
`CCDLibrary` the loader uses, so what it lists is what `mutate` accepts.

In [ ]:
library = CCDLibrary()
canonical = "ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split()

rows, mols, legends = [], [], []
for code in canonical:
    template = library[code][0]
    side_chain = template.atom_names - {"N", "CA", "C", "O", "OXT", "H", "H2", "HA", "HXT", "HA2", "HA3"}
    heavy = sum(1 for name in side_chain if template.name_to_atom[name].element != "H")
    target = "mutate to and from" if code not in ("GLY", "PRO") else (
        "to: adds HA; from: picks HA" if code == "GLY" else "refused (ring to backbone N)"
    )
    rows.append({"code": code, "name": template.description.title(), "side-chain heavy atoms": heavy,
                 "default formal charge": template.formal_charge, "as a mutation target": target})
    mols.append(residue_to_rdkit(fragment_from_ccd(code, "CA", library=library)))
    legends.append(f"{code}  ({template.description.title()})")

pd.DataFrame(rows).set_index("code")

In [ ]:
Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(230, 170), legends=legends)

Beyond the canonical 20, the CCD holds hundreds of peptide-linking components:
post-translational modifications, unnatural amino acids, and the D enantiomers.
A few that come up in labeling and structural work, fetched on demand:

In [ ]:
noncanonical = {
    "4II": "p-azido-L-phenylalanine (AzF)",
    "4AF": "p-acetyl-L-phenylalanine",
    "MSE": "selenomethionine",
    "SEP": "phosphoserine",
    "PTR": "phosphotyrosine",
    "ALY": "N6-acetyl-lysine",
    "MLY": "N6,N6-dimethyl-lysine",
    "NLE": "norleucine",
    "DPN": "D-phenylalanine",
    "DAL": "D-alanine",
}
online = CCDLibrary(download=True)
mols = [residue_to_rdkit(fragment_from_ccd(code, "CA", library=online)) for code in noncanonical]
Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(230, 170), legends=[f"{c}  ({n})" for c, n in noncanonical.items()])

## The mutated construct: S1381 → AzF, S1500 → Cys

Two residues need to be mutated. The residue numbers are already given in the file, so they are the fibronectin
numbers the papers use. Each call takes a few seconds on this 5,600-atom
protein, most of the time being the relaxation of the new side chain with everything else
held fixed.

In [ ]:
protein = mb.clone(wild_type)
backbone_before = {
    (resnum, name): protein.get_atom(resnum, name, chain_id="A").pos.copy()
    for resnum in (1381, 1500) for name in ("N", "CA", "C", "O", "H", "HA")
}

start = time.time()
azf = protein.mutate(1381, "4II", chain_id="A", relax=True)
cys = protein.mutate(1500, "CYS", chain_id="A", relax=True)
print(f"two mutations in {time.time() - start:.1f} s\n")

for residue in (azf, cys):
    print(f"{residue.name} {residue.resnum}: {residue.template.description}; formal charges {residue.atom_formal_charges or 'none'}; "
          f"HETATM={residue.hetatm}; alpha carbon is {alpha_carbon_cip(protein, residue.resnum, 'A')}")
    print("   atoms:", " ".join(sorted(p.name for p in residue.particles())))

moved = [key for key, pos in backbone_before.items()
         if abs(protein.get_atom(key[0], key[1], chain_id="A").pos - pos).max() > 1e-9]
print("\nbackbone atoms that moved:", moved or "none")
print("residues:", len(list(protein.residues())), "| inter-residue bond records:", protein.bond_records() or "none (a mutation is not a crosslink)")

The azide's charges (`N2` +1, `N3` −1) come from the CCD entry, so the residue
is neutral and the export below carries them. `4II` is not one of the 20
standard residues, so it is written as `HETATM` records under its CCD name, and
a residue library that knows the component reads it back with no hand-written
definition.

Cyan sticks are the new cysteine, green sticks the AzF:

In [ ]:
show_protein(protein, link_selection="1500:A", fragment_selection="1381:A")

## Chemistry the library does not know: the same side chain from SMILES

If the mutation you want to perform produces a structure that is NOT found in the CCD, write the side chain as a SMILES fragment with a `*` at the atom that bonds to the CA. The atom immediately attached to the `*` becomes the `CB`.

 Here, we showcase how to build the same AzF side chain so the two methods can be compared. Note: the modified residue doesn't have a template if there is no CCD definition, so you must write useful atom names and construct the template yourself so a downstream residue library can parse it. 

In [ ]:
from rdkit import Chem
from mbuild.biopolymers import draw_fragment

from_fragment = mb.clone(wild_type)
side_chain = prepare_fragment("*Cc1ccc(N=[N+]=[N-])cc1", "AZF")
draw_fragment(side_chain)

In [ ]:
residue = from_fragment.mutate(1381, side_chain, chain_id="A", relax=True)

print(f"{residue.name} {residue.resnum}: template={residue.template}; formal charges {residue.atom_formal_charges}; HETATM={residue.hetatm}")
print("   atoms:", " ".join(sorted(p.name for p in residue.particles())))
heavy = lambda r: sum(1 for p in r.particles() if p.element.symbol != "H")
print(f"heavy atoms: from fragment {heavy(residue)}, from CCD {heavy(azf)}; alpha carbon is {alpha_carbon_cip(from_fragment, 1381, 'A')}")

## A random chemistry not found in the CCD
The AzF fragment above had a CCD answer to compare against. Most custom
chemistry does not. Here the side chain is three fluorobenzene rings joined by
methylene carbons, which no CCD component and no force field residue library
describes. The workflow is the same: write the side chain as SMILES with `*`
on the atom that bonds to CA, hand it to `prepare_fragment`, and mutate.

`prepare_fragment` gives every atom the name it will carry in the residue and
in the written file, so `draw_fragment` is worth a look before going on: these
are the names you address later, when attaching to the side chain, describing
the residue to a downstream library, or picking atoms in a viewer. The `*` is
already replaced by a hydrogen, the one that leaves when the bond to CA forms,
and its neighbour is marked as the bond site.

In [ ]:
from mbuild.biopolymers import draw_fragment

TRIFLUORO = "*Cc1ccc(F)c(Cc2ccc(F)c(Cc3ccc(F)cc3)c2)c1"
side_chain = prepare_fragment(TRIFLUORO, "TFB")
print(f"{side_chain.name}: {side_chain.n_particles} atoms, formal charge {side_chain.formal_charge}, "
      f"bonds to CA through {', '.join(side_chain.link_atoms.values())}")
draw_fragment(side_chain)

The mutation call is the one used for AzF. The side chain has 41 atoms and
reaches well past where serine 1381 sat, so the relaxation has more to clear
and the call takes about a minute on a CPU. The residue takes the fragment's
name, is written as `HETATM` because it is not a standard residue, and has
no `template`, since no definition describes it: `formal_charge` and
`atom_formal_charges` still come from the SMILES.

In [ ]:
custom = mb.clone(wild_type)
start = time.time()
residue = custom.mutate(1381, side_chain, chain_id="A", relax=True)
print(f"mutation in {time.time() - start:.0f} s")
print(f"{residue.name} {residue.resnum}: {residue.n_particles} atoms; template={residue.template}; "
      f"HETATM={residue.hetatm}; formal charge {residue.formal_charge}; alpha carbon is {alpha_carbon_cip(custom, 1381, 'A')}")
print("   atoms:", " ".join(sorted(p.name for p in residue.particles())))
show_protein(custom, link_selection="1381:A", fragment_selection="1381:A")

The marked carbon of the fragment became `CB`, and the backbone atoms keep
the names and coordinates the file gave them, so the residue still reads as
part of the chain.

`save_pdb` writes it the same way as any non-standard residue: `HETATM`
records under the name `TFB`, and a `CONECT` line for every bond the residue
takes part in, including the peptide bonds to 1380 and 1382. The file is
complete as a structure. What no reader has is a *definition* of `TFB`, so a
template-matching reader, mBuild's `Protein` or OpenFF Pablo, refuses the
file until it is given one. The atoms, bonds, orders and charges that
definition needs are all on the mBuild residue, and notebook 4 shows how to
turn a residue into a Pablo `ResidueDefinition` with Pablo's public API.

In [ ]:
custom.save_pdb("1fnf_S1381TFB.pdb", overwrite=True)

lines = open("1fnf_S1381TFB.pdb").read().splitlines()
hetatm = [line for line in lines if line.startswith("HETATM") and line[17:20] == "TFB"]
serials = {int(line[6:11]) for line in hetatm}
conect = [line for line in lines if line.startswith("CONECT") and int(line[6:11]) in serials]
print(f"{len(hetatm)} HETATM records for TFB, {len(conect)} CONECT records starting from its atoms\n")
print("\n".join(hetatm[:6] + ["..."] + conect[:3]))

## Stereochemistry

A CCD code gives the mutant the handedness of the component, read from its
ideal coordinates, so `"CYS"` is L-cysteine and `"DCY"` would be D-cysteine.
`stereo="D"` or `stereo="L"` overrides that by swapping the places of the side
chain and the alpha hydrogen around CA. Two clones of the wild type, one call
each, side by side and centered on residue 1500. Cysteine's L form is *R* by
the CIP rules, because the sulfur outranks the carbonyl carbon; the geometric
handedness is what `mutate` controls.

The rule: `stereo` controls the geometry at CA, and the CIP label is a
name for that geometry that depends on what else is bonded. A CCD code
brings its own handedness; a fragment keeps the residue's.

In [ ]:
l_form, d_form = mb.clone(wild_type), mb.clone(wild_type)
l_form.mutate(1500, "CYS", chain_id="A", relax=True)
d_form.mutate(1500, "CYS", chain_id="A", stereo="D", relax=True)
print("CIP at CA 1500:  L-cysteine ->", alpha_carbon_cip(l_form, 1500, "A"), "  D-cysteine ->", alpha_carbon_cip(d_form, 1500, "A"))

views = [show_protein(p, link_selection="1500:A", fragment_selection="1500:A", width="440px", height="380px") for p in (l_form, d_form)]
ipywidgets.HBox(views)

## Export

`save_pdb` writes the mutant with the file's residue numbers, `4II` as
`HETATM` records, and `CONECT` records for every bond of the non-standard
residue. mBuild reads the file back, and OpenFF Pablo reads it once its CCD
cache may fetch the `4II` definition.

The same `save_pdb` rule as notebook 1, applied to a non-standard residue:
every bond of a `HETATM` residue is written as a CONECT record, because a
reader has no template to imply them from.

In [ ]:
protein.save_pdb("1fnf_S1381AzF_S1500C.pdb", overwrite=True)

again = Protein("1fnf_S1381AzF_S1500C.pdb", download=True)
print("mBuild reload:", again.get_residue(1381, chain_id="A").name, again.get_residue(1500, chain_id="A").name, again.n_particles, "atoms")

from openff.pablo import STD_CCD_CACHE, topology_from_pdb

STD_CCD_CACHE.auto_download = True
topology = topology_from_pdb("1fnf_S1381AzF_S1500C.pdb", residue_library=STD_CCD_CACHE)
azf_atoms = [a for a in topology.molecule(0).atoms if a.metadata["residue_name"] == "4II"]
print(f"Pablo: {topology.n_atoms} atoms, {topology.n_molecules} molecule, net charge {sum(a.formal_charge.m for a in topology.atoms)}; "
      f"4II has {len(azf_atoms)} atoms with charges {sorted((a.name, a.formal_charge.m) for a in azf_atoms if a.formal_charge.m)}")

## Recap, and what comes next

- The library decides what `mutate` knows. `library[code]` shows you.
- A fragment side chain uses the same call and leaves `template = None`.
- A residue with no template is written completely, but a template reader
  needs a definition before it reads the file back.

**Open questions.** Proline is refused in both directions because its side
chain bonds the backbone nitrogen. Whether to support it is a design
decision, not a limitation of the machinery. The hydrogen-renaming rule for
viewer-named hydrogens is a heuristic; a protein where it picks the wrong
heavy atom would be a useful test case.

The mutations exist so that dyes can be added in the next notebook: a click reaction on the azide
at 1381, which closes a triazole ring, and a thiol-Michael addition on the
cysteine at 1500, which moves a hydrogen. Neither is "replace one bond", so
neither fits `attach`'s leaving-atom keywords. Notebook 4 does both with
**reaction strings**, and then builds the residue definitions OpenFF Pablo
needs from the mBuild residues themselves.